# Kaggle / Colab runner

Clones the repo and runs the project scripts on a free GPU.

**Choose the phase** with `PHASE` in the first code cell: `3` = baseline with epoch-budget search, `4` = the baseline with 3 seeds plus the skin-tone analysis, `5` = the four image-processing settings (3 seeds each) and their comparison.

**Kaggle settings (right-hand panel):** Accelerator → *GPU T4 x2* or *GPU P100*; Internet → *On*
(needs a phone-verified account).

**Run it with *Save Version → Save & Run All (Commit)*,** not interactively. That runs in the background (up to 12 h, no idle timeout) and keeps everything in `/kaggle/working` as the version's output.

The repo and data live in a temporary folder (`/kaggle/temp`), so the ~3 GB of images are not saved as notebook output.
Each finished budget trial and CV fold is saved to `/kaggle/working/baseline_folds` (Phase 3) `/kaggle/working/seed_folds` (Phase 4) or `/kaggle/working/phase5_folds/<setting>` (Phase 5) straight away, so a crash or timeout keeps the finished folds. To resume, add the previous version's output as an input and set `RESUME_FROM` below; the training script skips the folds already saved.

Results are zipped into `/kaggle/working/phase<PHASE>_outputs.zip` at the end: download it and put it in the repo root.

In [1]:
import os, subprocess
REPO_URL = "https://github.com/sehersharik/skin-tone-triage.git"
WORK = "/kaggle/temp" if os.path.isdir("/kaggle") else "/content"   # Kaggle or Colab
os.makedirs(WORK, exist_ok=True)
OUTPUT = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"  # survives a timeout on Kaggle
PHASE = 5   # 3 = baseline + epoch budget, 4 = 3 seeds + skin-tone analysis, 5 = image-processing settings
FOLD_DIR = f"{OUTPUT}/" + {3: "baseline_folds", 4: "seed_folds", 5: "phase5_folds"}[PHASE]
RESUME_FROM = None   # e.g. "/kaggle/input/<previous-version-output>/seed_folds"
if RESUME_FROM and not os.path.isdir(FOLD_DIR):
    !cp -r {RESUME_FROM} {FOLD_DIR}
%cd {WORK}
!rm -rf skin-tone-triage
if subprocess.run(["git", "clone", "-q", REPO_URL]).returncode != 0:
    raise RuntimeError("Could not clone the repo. On Kaggle: Settings -> Internet -> On "
                       "(needs a phone-verified account), then run again.")
%cd skin-tone-triage
!git log --oneline -1
!pip install -q -r requirements.txt

/kaggle/temp


/kaggle/temp/skin-tone-triage
a88c904 (HEAD -> main, origin/main, origin/HEAD) Phase 5 setup: image-processing experiments (aug, Shades of Gray, CLAHE)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 7.4.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
notebook 6.5.7 requires jupyter-client<8,>=5.3.4, but you have jupyter-client 8.10.0 which is incompatible.
jupyter-kernel-gateway 2.5.2 requires jupyter_client<8.0,>=5.2.0, but you have jupyter-client 8.10.0 which is incompatible.


In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## Data: metadata, splits, and only the images we use

`build_metadata` stops if the splits differ from the committed `reports/split_fingerprint.txt`.

In [3]:
!python -m src.download_scin --config configs/data.yaml --images none
!python -m src.build_metadata --config configs/labels.yaml
!python -m src.download_scin --config configs/data.yaml --images used
!python -m src.prepare_images --config configs/baseline.yaml

dataset/scin_cases.csv: downloaded


dataset/scin_labels.csv: downloaded


dataset/scin_app_questions.csv: downloaded


dataset/scin_label_questions.csv: downloaded


images: 0it [00:00, ?it/s]
{'image_mode': 'none', 'images_referenced_in_cases_csv': 10380, 'images_requested': 0, 'downloaded': 0, 'skipped_existing': 0} failed=0
Run log: runs/20261102_221932_download_scin


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:1036: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


... (720 outputs left out)

resize:  99%|██████████████████████████████▊| 2515/2534 [01:10<00:00, 34.43it/s]

resize:  99%|██████████████████████████████▊| 2519/2534 [01:10<00:00, 35.25it/s]

resize: 100%|██████████████████████████████▉| 2524/2534 [01:10<00:00, 30.81it/s]

resize: 100%|███████████████████████████████| 2534/2534 [01:10<00:00, 35.83it/s]
{'resized': 2534}


## Phase 3: baseline 5-fold CV (test set untouched) and evaluation

In [4]:
if PHASE == 3:
    !python -m src.train --config configs/baseline.yaml --fold-outputs-dir {FOLD_DIR}
    !python -m src.evaluate --config configs/baseline.yaml

## Phase 4: 3 seeds on the same folds, then the skin-tone analysis (test set untouched)

About 4 min of training per seed on a T4. `image_factors` measures brightness, contrast, sharpness and resolution of every CV photo; `tone_analysis` writes `tone_analysis_cv` to `reports/metrics.json` plus the `phase4_*` figures.

In [5]:
if PHASE == 4:
    !python -m src.train --config configs/seeds.yaml --fold-outputs-dir {FOLD_DIR}
    !python -m src.image_factors --config configs/seeds.yaml
    !python -m src.tone_analysis --config configs/seeds.yaml

## Phase 5: image-processing settings, 3 seeds x 5 folds each (test set untouched)

For each setting the epoch budget is re-picked on fold 1 (seed 42), then 3 seeds x 5 folds are trained. Settings run one after another; every finished budget trial and fold is saved, so a rerun with `RESUME_FROM` continues where it stopped. `compare_settings` writes `phase5_cv` to `reports/metrics.json` and the `phase5_*` figures.

In [6]:
if PHASE == 5:
    for setting in ["none", "standard_aug", "shades_of_gray", "clahe"]:
        !python -m src.train --config configs/phase5/{setting}.yaml --fold-outputs-dir {FOLD_DIR}/{setting}
    !python -m src.compare_settings --config configs/phase5/compare.yaml

===== p5_none: seed 42 =====


Device: cuda  cases: 928  images: 2054  classes: ['dermatitis', 'folliculitis', 'insect_bite', 'psoriasis', 'tinea', 'urticaria']
Budget search: 5 epochs, scored on cv_fold 0


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


  0%|                                               | 0.00/20.5M [00:00<?, ?B/s]

 40%|███████████████                       | 8.12M/20.5M [00:00<00:00, 85.1MB/s]

100%|███████████████████████████████████████| 20.5M/20.5M [00:00<00:00, 143MB/s]


... (745 outputs left out)

Bootstrapping FST I-II ...


Bootstrapping FST III-IV ...


Bootstrapping FST V-VI ...



Epoch budget chosen on fold 1: {'p5_none': 5, 'p5_standard_aug': 15, 'p5_shades_of_gray': 5, 'p5_clahe': 5}
Overall, seed mean [95% CI]:
  p5_none            balanced_accuracy 0.394 [0.362, 0.424]  macro_f1 0.367 [0.337, 0.396]  macro_auc_ovr 0.761 [0.739, 0.781]
  p5_standard_aug    balanced_accuracy 0.417 [0.386, 0.449]  macro_f1 0.417 [0.384, 0.449]  macro_auc_ovr 0.800 [0.780, 0.820]
  p5_shades_of_gray  balanced_accuracy 0.409 [0.377, 0.442]  macro_f1 0.383 [0.352, 0.414]  macro_auc_ovr 0.756 [0.733, 0.777]
  p5_clahe           balanced_accuracy 0.405 [0.372, 0.436]  macro_f1 0.384 [0.352, 0.414]  macro_auc_ovr 0.764 [0.742, 0.785]
Paired change vs p5_none (outside the noise marked *):
  overall    p5_standard_aug    balanced_accuracy +0.023   macro_f1 +0.050*  macro_auc_ovr +0.039*  psoriasis_recall +0.111*  tinea_recall -0.082 
  overall    p5_shades_of_gray  balanced_accuracy +0.016   macro_f1 +0.016   macro_auc_ovr -0.005   psoriasis_recall +0.020   tinea_recall +0.007 
  ove

In [7]:
!pytest -q -p no:warnings
OUT = f"{OUTPUT}/phase{PHASE}_outputs.zip"
!zip -qr {OUT} reports docs/figures runs
import glob, json
for log in sorted(glob.glob({3: "reports/baseline_train_log.json", 4: "reports/seeds/*_train_log.json",
                                5: "reports/phase5/*_train_log.json"}[PHASE])):
    t = json.load(open(log))["timing"]
    print(log, "| GPU:", t["gpu"], "| seconds per CV fold:", t["seconds_per_cv_fold"])
print("Download:", OUT)

....

...

.

..

.......

..

... (6 outputs left out)

...

..

.                             [100%]
44 passed in 33.31s


reports/phase5/p5_clahe_seed42_train_log.json | GPU: Tesla T4 | seconds per CV fold: {'0': 52.5, '1': 52.0, '2': 52.6, '3': 52.7, '4': 52.5}
reports/phase5/p5_clahe_seed43_train_log.json | GPU: Tesla T4 | seconds per CV fold: {'0': 52.9, '1': 52.2, '2': 52.2, '3': 52.5, '4': 52.5}
reports/phase5/p5_clahe_seed44_train_log.json | GPU: Tesla T4 | seconds per CV fold: {'0': 53.1, '1': 52.0, '2': 52.5, '3': 51.8, '4': 52.6}
reports/phase5/p5_none_seed42_train_log.json | GPU: Tesla T4 | seconds per CV fold: {'0': 49.3, '1': 48.5, '2': 49.0, '3': 49.2, '4': 48.9}
reports/phase5/p5_none_seed43_train_log.json | GPU: Tesla T4 | seconds per CV fold: {'0': 49.1, '1': 48.8, '2': 49.1, '3': 49.1, '4': 48.9}
reports/phase5/p5_none_seed44_train_log.json | GPU: Tesla T4 | seconds per CV fold: {'0': 49.3, '1': 48.4, '2': 49.1, '3': 48.7, '4': 48.8}
reports/phase5/p5_shades_of_gray_seed42_train_log.json | GPU: Tesla T4 | seconds per CV fold: {'0': 54.4, '1': 54.0, '2': 53.4, '3': 55.1, '4': 54.0}
reports